# INSAT VIS & WV — Latent-Space 48-Frame Nowcasting

Runs the full 4-stage pipeline on a Colab GPU, for both channels:

1. **Encode** every TIFF into a cached SD-VAE latent
2. **Sanity check** — decode the latents and score PSNR/SSIM
3. **Train** the ConvLSTM RNN on latent sequences
4. **Forecast** a full 48-frame day and decode it to images

Set the runtime to **GPU** (Runtime -> Change runtime type -> T4 GPU).

## Setup

In [ ]:
# Point PROJECT_DIR at the repo root (the folder holding src/, vis/, wv/).
# Typical: upload the project to Google Drive, then mount it.
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/ISRO A.1'  # <-- edit to your path
%cd "$PROJECT_DIR"
!ls

In [ ]:
!pip install -q -r requirements.txt
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## Stage 1 — Encode images to cached latents

Writes one `.pt` per frame into `vis/latents/` and `wv/latents/`, plus a
`manifest.csv` per channel. Normalization stats are computed once and saved
back into each `config.yaml`. (Add `--limit 50` for a quick smoke test.)

In [ ]:
!python -m src.encode_latents --config vis/config.yaml

In [ ]:
!python -m src.encode_latents --config wv/config.yaml

## Stage 2 — VAE reconstruction sanity check

Target PSNR is ~30 dB. If it is far below, the pretrained SD-VAE is
out-of-distribution on INSAT imagery and should be fine-tuned before Stage 3.

In [ ]:
from IPython.display import Image, display
!python -m src.sanity_check --config vis/config.yaml
display(Image('vis/outputs/sanity_grid.png'))

In [ ]:
!python -m src.sanity_check --config wv/config.yaml
display(Image('wv/outputs/sanity_grid.png'))

## Stage 3 — Train the ConvLSTM RNN

VIS trains on daytime-only sequences; WV trains on full-day sequences.
Best weights are saved to `<ch>/checkpoints/best.pt`.

In [ ]:
!python -m src.train --config vis/config.yaml

In [ ]:
!python -m src.train --config wv/config.yaml

## Stage 4 — 48-frame day forecast

Autoregressive rollout -> decoded images. VIS = ~28 predicted daytime frames
+ ~20 dark night fills; WV = 48 predicted frames. Use `--date YYYYMMDD` to
pick a specific day.

In [ ]:
import glob
from IPython.display import Image, display
!python -m src.forecast --config vis/config.yaml
for p in sorted(glob.glob('vis/outputs/forecast_*_grid.png')):
    display(Image(p))

In [ ]:
!python -m src.forecast --config wv/config.yaml
for p in sorted(glob.glob('wv/outputs/forecast_*_grid.png')):
    display(Image(p))